# Семинар: базовые методы обработки текстов на естественном языке

Ноутбук основан на материалах лекции по теме «Введение в обработку текста на естественном языке» и учитывает следующие вопросы: расстояние Левенштейна, динамическое программирование, алгоритм Вагнера—Фишера, стемминг, лемматизация, стоп-слова, закон Ципфа, мешок слов, TF-IDF, косинусное сходство.

## Правила
1. В начале ноутбука задаётся `STUDENT_ID`
2. В ответах должны быть не только функции, но и выводы по конкретному варианту: таблицы, ранжирования, график Ципфа, анализ ошибок.
3. Быстрая генерация ответа внешним сервисом без запуска ноутбука недостаточна: часть баллов зависит от индивидуальных численных результатов и контрольной подписи.
4. При решении действуют следующие ограничения:
    - Нельзя использовать готовые функции `edit_distance`, `TfidfVectorizer`, `CountVectorizer`, `cosine_similarity` для основных заданий. Их можно использовать только для дополнительной проверки после собственной реализации.
    - Разрешены стандартная библиотека Python, `numpy`, `pandas`, `matplotlib`.
    - Для сдачи нужно выполнить все ячейки сверху вниз и оставить видимыми результаты ключевых вычислений.

## Структура
- Первая пара: задания 1–10.
- Вторая пара: задания 11–20.

In [ ]:
# Впишите свой идентификатор свой идентификатор.
STUDENT_ID = "123456"  # например: "123456" из "123456@edu.fa.ru"

In [ ]:

import re
import math
import json
import hashlib
import random
from collections import Counter, defaultdict

import numpy as np

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None


BASE_DOCUMENTS = [
    {"topic": "space", "text": "Российские инженеры испытали новый спутниковый модуль связи для мониторинга климата и ледовой обстановки."},
    {"topic": "space", "text": "Частная космическая компания перенесла запуск ракеты из-за сильного ветра и проверки системы навигации."},
    {"topic": "space", "text": "Астрономы обнаружили необычный сигнал от далекой галактики и подготовили серию повторных наблюдений."},
    {"topic": "space", "text": "Орбитальная станция получила обновление программного обеспечения для автоматической стыковки грузового корабля."},
    {"topic": "sport", "text": "Футбольный клуб усилил защиту перед решающим матчем чемпионата и провел открытую тренировку."},
    {"topic": "sport", "text": "Теннисистка выиграла финал турнира после трех сетов и поднялась на четвертую позицию рейтинга."},
    {"topic": "sport", "text": "Тренер сборной объяснил поражение высокой нагрузкой игроков и ошибками при быстрых контратаках."},
    {"topic": "sport", "text": "Юные пловцы установили рекорд области на дистанции сто метров вольным стилем."},
    {"topic": "economy", "text": "Центральный банк сохранил ключевую ставку и отметил замедление инфляции в промышленном секторе."},
    {"topic": "economy", "text": "Экспортеры увеличили поставки оборудования после снижения логистических издержек и роста внешнего спроса."},
    {"topic": "economy", "text": "Биржевой индекс снизился на фоне слабой отчетности технологических компаний и падения цен на сырье."},
    {"topic": "economy", "text": "Малый бизнес получил новые налоговые льготы для инвестиций в цифровые сервисы и обучение сотрудников."},
    {"topic": "medicine", "text": "Врачи городской больницы внедрили систему раннего выявления осложнений на основе анализа лабораторных показателей."},
    {"topic": "medicine", "text": "Исследователи сравнили эффективность двух вакцин и опубликовали данные о частоте побочных реакций."},
    {"topic": "medicine", "text": "Клиника открыла отделение телемедицины для консультаций пациентов из удаленных населенных пунктов."},
    {"topic": "medicine", "text": "Новый протокол реабилитации помог пациентам быстрее восстановить подвижность после операций на суставах."},
    {"topic": "nlp", "text": "Алгоритм токенизации разделил текст на слова и удалил служебные символы перед построением частотной модели."},
    {"topic": "nlp", "text": "Модель мешка слов игнорирует порядок токенов, но позволяет быстро сравнивать документы по словарю терминов."},
    {"topic": "nlp", "text": "Лемматизация уменьшила число различных словоформ и повысила качество поиска похожих фрагментов."},
    {"topic": "nlp", "text": "TF IDF выделил редкие информативные термы и снизил вес частотных слов в новостном корпусе."},
    {"topic": "education", "text": "Студенты реализовали динамическое программирование и сравнили сложность наивного и табличного алгоритмов."},
    {"topic": "education", "text": "Преподаватель подготовил контрольные варианты с индивидуальными наборами данных и автоматическими проверками."},
    {"topic": "education", "text": "Семинар завершился обсуждением ошибок в коде, влияния нормализации и интерпретации численных результатов."},
    {"topic": "education", "text": "Команда разработала учебный проект, где поиск документов оценивается по полноте и точности ранжирования."},
]

BASE_WORDS = [
    "алгоритм", "вектор", "корпус", "лемматизация", "стемминг", "токенизация",
    "документ", "частота", "косинус", "расстояние", "матрица", "редакция",
    "модель", "словарь", "поиск", "индексация", "термин", "нормализация",
    "галактика", "ракета", "спортсмен", "инфляция", "пациент", "студент",
    "программа", "ошибка", "система", "данные", "анализ", "сходство"
]

MINI_STOP_WORDS = {
    "и", "в", "во", "на", "по", "с", "со", "для", "из", "за", "от", "до",
    "о", "об", "а", "но", "что", "как", "это", "после", "перед", "при",
    "или", "не", "быстро", "новый", "новые", "нового", "свой", "свои"
}

SUFFIXES = [
    "иями", "ями", "ами", "его", "ого", "ему", "ому", "ыми", "ими",
    "ая", "яя", "ое", "ее", "ые", "ие", "ый", "ий", "ой",
    "ах", "ях", "ам", "ям", "ом", "ем", "ов", "ев",
    "а", "я", "ы", "и", "е", "у", "ю", "о"
]

LEMMA_MAP = {
    "слова": "слово", "слов": "слово", "словом": "слово", "словами": "слово",
    "документы": "документ", "документа": "документ", "документами": "документ",
    "тексты": "текст", "текста": "текст", "текстом": "текст",
    "данных": "данные", "данными": "данные",
    "модели": "модель", "моделью": "модель",
    "стали": "стать", "сталью": "сталь",
    "был": "быть", "была": "быть", "были": "быть",
    "пациенты": "пациент", "пациентам": "пациент",
    "студенты": "студент", "студентов": "студент",
    "системы": "система", "системой": "система",
    "ошибки": "ошибка", "ошибками": "ошибка",
    "термы": "терм", "термов": "терм",
}

CYR = "абвгдеежзийклмнопрстуфхцчшщьыэюя"


def _seed_from_id(student_id: str) -> int:
    return int(hashlib.sha256(student_id.encode("utf-8")).hexdigest()[:16], 16)


def _corrupt_once(word: str, rng: random.Random) -> str:
    if len(word) < 4:
        return word + rng.choice(CYR)
    op = rng.choice(["delete", "insert", "replace", "transpose"])
    i = rng.randrange(len(word))
    if op == "delete":
        return word[:i] + word[i + 1:]
    if op == "insert":
        return word[:i] + rng.choice(CYR) + word[i:]
    if op == "replace":
        ch = rng.choice(CYR)
        if ch == word[i]:
            ch = rng.choice(CYR)
        return word[:i] + ch + word[i + 1:]
    if op == "transpose" and len(word) > 3:
        i = rng.randrange(len(word) - 1)
        return word[:i] + word[i + 1] + word[i] + word[i + 2:]
    return word


def build_variant(student_id: str) -> dict:
    rng = random.Random(_seed_from_id(student_id))
    docs = rng.sample(BASE_DOCUMENTS, 14)
    target_topic = rng.choice(sorted({d["topic"] for d in docs}))
    topic_docs = [d for d in docs if d["topic"] == target_topic]
    if not topic_docs:
        topic_docs = docs[:2]
    query_base = rng.choice(topic_docs)["text"]
    query = " ".join(query_base.split()[:12]) + " похожие документы поиск"

    words = rng.sample(BASE_WORDS, 12)
    noisy_pairs = []
    for w in words:
        noisy = _corrupt_once(w, rng)
        if noisy == w:
            noisy = _corrupt_once(w, rng)
        noisy_pairs.append({"clean": w, "noisy": noisy})

    injected_terms = rng.sample(BASE_WORDS, 4)
    docs_texts = []
    for i, d in enumerate(docs):
        extra = " ".join(rng.sample(injected_terms, k=rng.randint(1, 2)))
        docs_texts.append(f"{d['text']} {extra}")

    return {
        "student_id": student_id,
        "variant_hash": hashlib.sha256((student_id + "::nlp07").encode("utf-8")).hexdigest()[:12],
        "documents": docs_texts,
        "topics": [d["topic"] for d in docs],
        "query": query,
        "noisy_pairs": noisy_pairs,
        "dictionary": sorted(set(BASE_WORDS + [p["clean"] for p in noisy_pairs])),
        "stop_words_base": sorted(MINI_STOP_WORDS),
        "suffixes": SUFFIXES,
        "lemma_map": LEMMA_MAP,
        "control_terms": injected_terms,
    }


variant = build_variant(STUDENT_ID)
print("STUDENT_ID:", STUDENT_ID)
print("variant_hash:", variant["variant_hash"])
print("documents:", len(variant["documents"]))
print("query:", variant["query"])
print("first noisy pairs:", variant["noisy_pairs"][:3])


STUDENT_ID: 123456
variant_hash: e80a483deffa
documents: 14
query: Биржевой индекс снизился на фоне слабой отчетности технологических компаний и падения цен похожие документы поиск
first noisy pairs: [{'clean': 'редакция', 'noisy': 'реакция'}, {'clean': 'матрица', 'noisy': 'мартица'}, {'clean': 'данные', 'noisy': 'днаные'}]


## Решения

## Задание 01. Запустите ячейку ниже без изменений его содержимого

Сгенерируется индивидуальный вариант и выведется хеш варианта задания, количество документов, список тем и первые 5 пар из набора данных.


In [ ]:
print("variant_hash:", variant["variant_hash"])
print("n_documents:", len(variant["documents"]))
print("topics:", variant["topics"])
print("first 5 pairs:")
for p in variant["noisy_pairs"][:5]:
    print(p)

## Задание 02. Токенизация русского текста

Реализуйте функцию `tokenize_ru(text, keep_numbers=False)`, которая:
- приводит текст к нижнему регистру;
- заменяет `ё` на `е`;
- оставляет только русские слова;
- при `keep_numbers=True` также оставляет числа.

Проверьте функцию на первом документе и на запросе из варианта.

### Решение

In [ ]:
def tokenize_ru(text: str, keep_numbers: bool = False) -> list[str]:
    # ВАШ КОД

tokens_doc0 = tokenize_ru(variant["documents"][0])
tokens_query = tokenize_ru(variant["query"])
print(tokens_doc0[:30])
print(tokens_query)
assert all(t == t.lower() for t in tokens_doc0)
assert all(re.fullmatch(r"[а-я]+", t) for t in tokens_doc0)

## Задание 03. Частотный словарь документа

Реализуйте `term_frequencies(tokens)`. Для первого документа выведите:
- число токенов;
- число различных токенов;
- 10 самых частотных токенов.

Сделайте вывод: какие токены являются содержательными, а какие похожи на шумовые/служебные.

### Решение

In [ ]:
def term_frequencies(tokens: list[str]) -> Counter:
    # ВАШ КОД

freq0 = term_frequencies(tokens_doc0)
print("N =", sum(freq0.values()))
print("B =", len(freq0))
print(freq0.most_common(10))

content_like = [w for w, c in freq0.most_common(10) if w not in MINI_STOP_WORDS]
noise_like = [w for w, c in freq0.most_common(10) if w in MINI_STOP_WORDS]
print("Содержательные кандидаты:", content_like)
print("Служебные/шумовые кандидаты:", noise_like)

## Задание 04. Расстояние Левенштейна через динамическое программирование

Реализуйте алгоритм Вагнера—Фишера: `levenshtein_matrix(s1, s2, insert_cost=1, delete_cost=1, substitute_cost=1)`.
Функция должна возвращать матрицу размера `(len(s1)+1, len(s2)+1)`.

Проверьте функцию на первой паре `noisy -> clean`. Выведите расстояние и матрицу.

### Решение

In [ ]:
def levenshtein_matrix(
    s1: str,
    s2: str,
    insert_cost: float = 1,
    delete_cost: float = 1,
    substitute_cost: float = 1,
    substitute_cost_fn=None,
) -> list[list[float]]:
    # ВАШ КОД

pair0 = variant["noisy_pairs"][0]
D0 = levenshtein_matrix(pair0["noisy"], pair0["clean"])
print(pair0)
print("distance =", D0[-1][-1])
print(np.array(D0))
assert len(D0) == len(pair0["noisy"]) + 1
assert len(D0[0]) == len(pair0["clean"]) + 1

## Задание 05. Редакционное предписание

По матрице из задания 4 восстановите одно оптимальное редакционное предписание.

Реализуйте:
- `edit_script(s1, s2, D, ...)`;
- `apply_edit_script(script)`, возвращающую строку-результат.

Проверьте, что применение предписания к `noisy` дает `clean`.

### Решение

In [ ]:
def edit_script(
    s1: str,
    s2: str,
    D: list[list[float]],
    insert_cost: float = 1,
    delete_cost: float = 1,
    substitute_cost: float = 1,
    substitute_cost_fn=None,
) -> list[tuple[str, str, str]]:
    # формат операции: ("M/R/I/D", source_char, target_char)
    # ВАШ КОД
    raise NotImplementedError

def apply_edit_script(script: list[tuple[str, str, str]]) -> str:
    # ВАШ КОД
    raise NotImplementedError

script0 = edit_script(pair0["noisy"], pair0["clean"], D0)
print(script0)
print(apply_edit_script(script0), "==", pair0["clean"])
assert apply_edit_script(script0) == pair0["clean"]

## Задание 06. Расстояние Дамерау—Левенштейна для соседних транспозиций

Реализуйте вариант Damerau-Levenshtein OSA, где дополнительно разрешена транспозиция двух соседних символов.
Сравните расстояния Левенштейна и Дамерау—Левенштейна для всех пар `noisy/clean`.
Найдите пары, где транспозиция уменьшила расстояние.

### Решение

In [ ]:
def damerau_osa_distance(s1: str, s2: str, substitute_cost: float = 1) -> float:
    # ВАШ КОД

rows = []
for p in variant["noisy_pairs"]:
    lev = levenshtein_matrix(p["noisy"], p["clean"])[-1][-1]
    dam = damerau_osa_distance(p["noisy"], p["clean"])
    rows.append((p["noisy"], p["clean"], lev, dam, dam < lev))

rows

[('реакция', 'редакция', 1.0, 1.0, False),
 ('мартица', 'матрица', 2.0, 1.0, True),
 ('днаные', 'данные', 2.0, 1.0, True),
 ('слоарь', 'словарь', 1.0, 1.0, False),
 ('спациент', 'пациент', 1.0, 1, False),
 ('нормализацие', 'нормализация', 1.0, 1.0, False),
 ('моюдель', 'модель', 1.0, 1.0, False),
 ('систеа', 'система', 1.0, 1.0, False),
 ('этудент', 'студент', 1.0, 1.0, False),
 ('аанлиз', 'анализ', 2.0, 1.0, True),
 ('вектр', 'вектор', 1.0, 1.0, False),
 ('частцта', 'частота', 1.0, 1.0, False)]

## Задание 07. Коррекция опечаток по ближайшему слову

Используя расстояние Левенштейна, реализуйте `nearest_word(noisy, dictionary)`.
Требования:
- ранжировать по нормированному расстоянию `distance / max(len(noisy), len(candidate))`;
- при равенстве выбирать лексикографически меньший кандидат.

Оцените accuracy на индивидуальных парах `noisy/clean`.

### Решение

In [ ]:
def nearest_word(noisy: str, dictionary: list[str]) -> tuple[str, float]:
    # ВАШ КОД

predictions = []
for p in variant["noisy_pairs"]:
    pred, score = nearest_word(p["noisy"], variant["dictionary"])
    predictions.append((p["noisy"], p["clean"], pred, round(score, 3), pred == p["clean"]))

accuracy = sum(ok for *_, ok in predictions) / len(predictions)
print("accuracy:", accuracy)
predictions

## Задание 08. Взвешенное редакционное расстояние

Добавьте функцию стоимости замены `weighted_substitution(a, b)`, где похожие пары символов имеют меньший штраф:
`е/ё`, `и/й`, `о/а`, `т/д`, `с/з`.

Сравните обычное и взвешенное расстояние для индивидуальных пар. Укажите 2 случая, где взвешивание изменяет интерпретацию близости.

### Решение

In [ ]:
def weighted_substitution(a: str, b: str) -> float:
    # ВАШ КОД

weighted_rows = []
for p in variant["noisy_pairs"]:
    d_uniform = levenshtein_matrix(p["noisy"], p["clean"])[-1][-1]
    d_weighted = levenshtein_matrix(
        p["noisy"], p["clean"], substitute_cost_fn=weighted_substitution
    )[-1][-1]
    weighted_rows.append((p["noisy"], p["clean"], d_uniform, d_weighted, d_weighted < d_uniform))

weighted_rows

## Задание 09. Простой стеммер на правилах

Реализуйте `simple_stem(word, suffixes, min_len=3)`, который отрезает самый длинный подходящий суффикс.
Примените стеммер к токенам корпуса.
Покажите 10 примеров: `слово -> стем`.
Найдите не менее 3 коллизий, когда разные слова дали один и тот же стем.

### Решение

In [ ]:
def simple_stem(word: str, suffixes: list[str], min_len: int = 3) -> str:
    # ВАШ КОД

all_tokens = [t for doc in variant["documents"] for t in tokenize_ru(doc)]
stem_examples = [(w, simple_stem(w, variant["suffixes"])) for w in all_tokens[:30]]
print(stem_examples)

stem_to_words = defaultdict(set)
for w in set(all_tokens):
    stem_to_words[simple_stem(w, variant["suffixes"])].add(w)

collisions = {stem: sorted(words) for stem, words in stem_to_words.items() if len(words) > 1}
list(collisions.items())[:10]

## Задание 10. Лемматизация на мини-словаре и эвристиках

Реализуйте `normalize_token(token, lemma_map, suffixes)`.
Сначала используйте словарь лемм, затем fallback на `simple_stem`.

Сравните стемминг и нормализацию на 30 токенах корпуса. Найдите случаи, когда словарная нормализация лучше простого отрезания суффикса.

### Решение

In [ ]:
def normalize_token(token: str, lemma_map: dict[str, str], suffixes: list[str]) -> str:
    # ВАШ КОД

norm_examples = [
    (w, simple_stem(w, variant["suffixes"]), normalize_token(w, variant["lemma_map"], variant["suffixes"]))
    for w in all_tokens[:50]
]
norm_examples[:30]

## Задание 11. Стоп-слова и корпусно-зависимый список

Реализуйте `build_stop_words(texts, base_stop_words, top_k=10)`.
Функция должна объединять базовый список стоп-слов и `top_k` самых частотных слов корпуса.

Сравните топ-20 частотных слов до и после удаления стоп-слов. Объясните, что изменилось.

### Решение

In [ ]:
def build_stop_words(texts: list[str], base_stop_words: list[str], top_k: int = 10) -> set[str]:
    # ВАШ КОД

stop_words = build_stop_words(variant["documents"], variant["stop_words_base"], top_k=10)
tokens_wo_stop = [t for t in all_tokens if t not in stop_words]
print("stop_words sample:", sorted(stop_words)[:30])
print("before:", Counter(all_tokens).most_common(20))
print("after:", Counter(tokens_wo_stop).most_common(20))

## Задание 12. Проверка закона Ципфа на индивидуальном корпусе

Постройте таблицу `rank, term, frequency` для полного набора токенов корпуса.
Оцените наклон прямой в координатах `log(rank)` — `log(frequency)` через `np.polyfit`.
Постройте график с помощью `matplotlib`.

Сделайте вывод: насколько распределение похоже на закон Ципфа и почему малый корпус может искажать результат.

### Решение

In [ ]:
def zipf_table(tokens: list[str]) -> list[tuple[int, str, int]]:
    # ВАШ КОД

zt = zipf_table(all_tokens)
print(zt[:15])

# ВАШ КОД для графика

## Задание 13. Словарь и матрица Bag-of-Words

Реализуйте:
- `preprocess_document(text, stop_words=None, mode="raw")`;
- `build_vocabulary(tokenized_docs, min_df=1)`;
- `bow_matrix(tokenized_docs, vocabulary)`.

Постройте BoW-матрицу для документов варианта после удаления стоп-слов и нормализации.

### Решение

In [ ]:
def preprocess_document(text: str, stop_words: set[str] | None = None, mode: str = "raw") -> list[str]:
    # mode: "raw", "stem", "lemma"
    # ВАШ КОД

def build_vocabulary(tokenized_docs: list[list[str]], min_df: int = 1) -> dict[str, int]:
   # ВАШ КОД

def bow_matrix(tokenized_docs: list[list[str]], vocabulary: dict[str, int]) -> np.ndarray:
    # ВАШ КОД

docs_tokens = [preprocess_document(d, stop_words=stop_words, mode="lemma") for d in variant["documents"]]
vocab = build_vocabulary(docs_tokens, min_df=1)
X_bow = bow_matrix(docs_tokens, vocab)
print(X_bow.shape)
print(list(vocab.items())[:20])
assert X_bow.shape == (len(variant["documents"]), len(vocab))

## Задание 14. Булевские веса, TF и sublinear TF

Реализуйте три схемы взвешивания:
- boolean;
- raw term frequency;
- sublinear TF: `1 + log(tf)` для `tf > 0`.

Для первых трех документов сравните количество ненулевых признаков и сумму весов.

### Решение

In [ ]:
def boolean_weight(X: np.ndarray) -> np.ndarray:
    # ВАШ КОД

def raw_tf_weight(X: np.ndarray) -> np.ndarray:
    # ВАШ КОД

def sublinear_tf_weight(X: np.ndarray) -> np.ndarray:
    # ВАШ КОД

for name, W in [
    ("bool", boolean_weight(X_bow)),
    ("tf", raw_tf_weight(X_bow)),
    ("sublinear", sublinear_tf_weight(X_bow)),
]:
    print(name, "nnz first3:", (W[:3] > 0).sum(axis=1), "sum first3:", np.round(W[:3].sum(axis=1), 3))

## Задание 15. TF-IDF вручную

Реализуйте smooth IDF:

`idf(t) = log((1 + N) / (1 + df(t))) + 1`

Постройте TF-IDF на основе sublinear TF. Для каждого из первых 3 документов выведите топ-5 терминов по TF-IDF.

### Решение

In [ ]:
def smooth_idf(X: np.ndarray) -> np.ndarray:
    # ВАШ КОД

def tfidf_matrix(X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    # ВАШ КОД

X_tfidf, idf = tfidf_matrix(X_bow)
inv_vocab = {i: t for t, i in vocab.items()}

def top_terms(row: np.ndarray, k: int = 5):
    idx = np.argsort(-row)[:k]
    return [(inv_vocab[i], float(row[i])) for i in idx if row[i] > 0]

for i in range(min(3, X_tfidf.shape[0])):
    print("doc", i, variant["topics"][i], top_terms(X_tfidf[i], 5))

## Задание 16. Косинусное сходство

Реализуйте `cosine_matrix(X)` без использования готовой функции.
Проверьте:
- диагональ равна 1 для ненулевых документов;
- матрица симметрична;
- значения лежат в диапазоне `[0, 1]`.

Выведите наиболее похожую пару разных документов.

### Решение

In [ ]:
def cosine_matrix(X: np.ndarray) -> np.ndarray:
    # ВАШ КОД

S = cosine_matrix(X_tfidf)
print(np.round(S[:5, :5], 3))

nonzero_docs = np.linalg.norm(X_tfidf, axis=1) > 0
assert np.allclose(np.diag(S)[nonzero_docs], 1.0)
assert np.allclose(S, S.T)
assert np.nanmin(S) >= -1e-9 and np.nanmax(S) <= 1 + 1e-9

# ВАШ КОД для вычисления наиболее похожей пары разных документов
print("most similar pair:", i, j, "score:", S[i, j])
print("topics:", variant["topics"][i], variant["topics"][j])
print("doc_i:", variant["documents"][i])
print("doc_j:", variant["documents"][j])

## Задание 17. Поиск похожих документов по запросу

Постройте TF-IDF-вектор запроса `variant["query"]` в том же словаре.
Реализуйте `rank_documents(query, ...)`, которая возвращает топ-5 документов по косинусному сходству.
Выведите номер документа, тему, score и текст.

### Решение

In [ ]:
def transform_query(query: str, vocabulary: dict[str, int], stop_words: set[str], mode: str = "lemma") -> np.ndarray:
    # ВАШ КОД

def rank_documents(query: str, X_docs: np.ndarray, vocabulary: dict[str, int], stop_words: set[str], mode: str = "lemma", top_k: int = 5):
    # ВАШ КОД

ranking = rank_documents(variant["query"], X_tfidf, vocab, stop_words, mode="lemma", top_k=5)
ranking

## Задание 18. Сравнение трех пайплайнов предобработки

Сравните пайплайны:
1. `raw`: только токенизация;
2. `nostop`: токенизация + стоп-слова;
3. `lemma`: токенизация + стоп-слова + нормализация.

Для каждого пайплайна выведите:
- размер словаря;
- топ-3 найденных документа;
- совпадает ли тема первого результата с темой запроса.

### Решение

In [ ]:
def evaluate_pipeline(mode_name: str):
    # ВАШ КОД

pipeline_results = {}
for mode_name in ["raw", "nostop", "lemma"]:
    pipeline_results[mode_name] = evaluate_pipeline(mode_name)

pipeline_results

## Задание 19. Анализ ошибок и ограничений

На основе результатов заданий 7, 12, 17, 18 напишите краткий анализ:
- 3 конкретные ошибки или слабые места;
- почему они возникли;
- как их можно исправить технически.

Ответ должен ссылаться на численные результаты вашего варианта.

### Решение

In [ ]:
# Напишите анализ в Markdown-ячейке ниже или сформируйте текст программно.

## Задание 20. Выводы

Сформируйте выводы, которые должны содержать:
- accuracy коррекции опечаток;
- наклона Zipf;
- индексов топ-5 документов в поиске;
- размера словаря финального пайплайна;
- краткий итоговый отчет по работе.

### Решение

In [ ]:
# Напишите выводы в Markdown-ячейке ниже или сформируйте текст программно.